# Sztuczne sieci neuronowe i głębokie uczenie - Sprawozdanie
## Laboratorium nr 8: Praca z korpusami językowymi
 **Imię i nazwisko: Aleksander Rak**
**Wykorzystane narzędzia:** Python 3, Jupyter Notebook / Google Colab, `datasets`, `transformers`, `pandas`, `matplotlib`, `seaborn`, `stop_words`

---

### Zadanie 1: Pobieranie i eksploracja korpusu tekstowego

Wykorzystujemy bibliotekę `datasets` do pobrania polskojęzycznego fragmentu bazy danych artykułów Wikipedii (`20220301.pl`). Wyodrębniamy podzbiór składający się z pierwszych 200 artykułów do dalszych analiz inżynierii lingwistycznej.

In [ ]:
from datasets import load_dataset

print("Pobieranie korpusu i wyodrębnianie 200 artykułów...")
wiki_dataset = load_dataset("wikipedia", "20220301.pl", split="train", trust_remote_code=True)
articles = [wiki_dataset[i]["text"] for i in range(200)]

print(f"Pomyślnie załadowano {len(articles)} artykułów.")
print(f"Przykładowy początek pierwszego artykułu:\n{articles[0][:150]}...")

### Zadanie 2: Podstawowe statystyki tekstu

W tej sekcji wyliczamy podstawowe metryki długości dokumentów (w znakach i w słowach) oraz generujemy histogramy rozkładów empirycznych.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

char_counts = [len(art) for art in articles]
word_counts = [len(art.split()) for art in articles]

df_stats = pd.DataFrame({
    'Liczba znaków': char_counts,
    'Liczba słów': word_counts
})

print("--- PODSTAWOWE STATYSTYKI KORPUSU ---")
print(df_stats.describe().loc[['mean', '50%', 'min', 'max']])

# Generowanie wykresów rozkładu
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(char_counts, bins=25, ax=axes[0], color='skyblue', kde=True)
axes[0].set_title('Rozkład liczby znaków w artykułach')
axes[0].set_xlabel('Liczba znaków')
axes[0].set_ylabel('Częstość')

sns.histplot(word_counts, bins=25, ax=axes[1], color='salmon', kde=True)
axes[1].set_title('Rozkład liczby słów w artykułach')
axes[1].set_xlabel('Liczba słów')
axes[1].set_ylabel('Częstość')

plt.tight_layout()
plt.show()

### Zadanie 3: Czyszczenie tekstu i analiza częstotliwości (Prawo Zipfa)

Tekst wejściowy jest normalizowany do małych liter oraz czyszczony przy użyciu wyrażeń regularnych (`regex`) w celu eliminacji znaków interpunkcyjnych, cyfr i znaczników formatowania.

In [ ]:
import re
from collections import Counter

all_tokens = []
for art in articles:
    # Zamiana na małe litery
    clean_text = art.lower()
    # Usuwanie znaków interpunkcyjnych i cyfr przy użyciu wyrażenia regularnego
    clean_text = re.sub(r'[^\a-zćłóśźżąęń\s]', '', clean_text)
    # Tokenizacja po białych znakach
    tokens = clean_text.split()
    all_tokens.extend(tokens)

word_freq = Counter(all_tokens)
top_30 = word_freq.most_common(30)

print("Top 10 najczęstszych słów:")
for word, freq in top_30[:10]:
    print(f"{word:<10} : {freq}")

# Wizualizacja Prawa Zipfa
frequencies = [freq for _, freq in word_freq.most_common()]
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(frequencies) + 1), frequencies, color='purple', linewidth=2)
plt.title('Wykres częstotliwości słów (Prawo Zipfa)')
plt.xlabel('Ranga słowa (log)')
plt.ylabel('Częstotliwość występowania (log)')
plt.xscale('log')
plt.yscale('log')
plt.grid(True, which="both", ls="--")
plt.show()

### Zadanie 4: Tokenizacja polska a angielska (HerBERT vs mBERT)

Polszczyzna cechuje się bardzo bogatą fleksją i morfologią. W tym punkcie badamy, jak dedykowany model polski (`allegro/herbert-base-cased`) radzi sobie z podziałem wyrazów na podsłowa w porównaniu z modelem wielojęzycznym (`bert-base-multilingual-cased`). Wykorzystujemy do tego 10 zdań wyekstrahowanych z korpusu.

In [ ]:
from transformers import AutoTokenizer
import pandas as pd

# Załadowanie tokenizatorów
herbert_tok = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
mbert_tok = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

# Przykładowe zdania z tekstu
sample_sentences = [
    "Włodzimierz urodził się w małej podkarpackiej miejscowości.",
    "Rzeczypospolita Polska jest demokratycznym państwem prawnym.",
    "Najważniejszym elementem architektury transformer jest warstwa uwagi.",
    "Programowanie strukturalne ustąpiło miejsca paradygmatom obiektowym.",
    "Konwolucyjne sieci neuronowe doskonale radzą sobie z obrazami.",
    "Morfologia języka polskiego stanowi ogromne wyzwanie inżynieryjne.",
    "Zbiory danych tekstowych pobieramy bezpośrednio z platformy HuggingFace.",
    "Algorytmy n-gramowe pozwalają na wyznaczenie najczęstszych kolokacji.",
    "Wielojęzyczne modele językowe często cierpią na tak zwane zjawisko klątwy wymiaru.",
    "Analiza statystyczna tekstu pozwala zaobserwować empiryczne działanie prawa Zipfa."
]

data_records = []
for i, sent in enumerate(sample_sentences):
    h_tokens = herbert_tok.tokenize(sent)
    m_tokens = mbert_tok.tokenize(sent)
    data_records.append({
        "ID": f"Z{i+1}",
        "Zdanie": sent[:40] + "...",
        "herbert_n": len(h_tokens),
        "mbert_n": len(m_tokens),
        "roznica": len(m_tokens) - len(h_tokens)
    })

df_tok = pd.DataFrame(data_records)
print(df_tok.to_string(index=False))

# Wykres słupkowy porównawczy
plt.figure(figsize=(12, 6))
x_indices = np.arange(len(df_tok))
bar_width = 0.35

plt.bar(x_indices - bar_width/2, df_tok['herbert_n'], bar_width, label='HerBERT (Dedykowany PL)', color='teal')
plt.bar(x_indices + bar_width/2, df_tok['mbert_n'], bar_width, label='mBERT (Wielojęzyczny)', color='orange')

plt.title('Porównanie liczby subtokenów: HerBERT vs mBERT')
plt.xlabel('Indeks zdania')
plt.ylabel('Liczba wygenerowanych tokenów')
plt.xticks(x_indices, df_tok['ID'])
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

#### Analiza i interpretacja lingwistyczna:

* **Który model jest bardziej efektywny dla języka polskiego?**
  Model **HerBERT** jest znacznie bardziej efektywny. Generuje średnio mniejszą liczbę subtokenów dla tego samego tekstu, ponieważ jego słownik subwyrazów (*Vocabulary*) został zbudowany wyłącznie na korpusach polskojęzycznych. Dzięki temu całe rzadkie wyrazy lub rdzenie fleksyjne (np. `-Państwem`, `-strukturalne`) nie są nadmiernie szatkowane na pojedyncze sylaby lub znaki.

* **Co oznacza większy/mniejszy słownik subsłów dla jakości modelu?**
  Większy i dedykowany słownik subsłów sprawia, że model lepiej agreguje semantykę i kontekst całego słowa, drastycznie skracając efektywną długość sekwencji wejściowej przesyłanej do warstw *Self-Attention*. Z kolei mniejszy dedykowany słownik (lub słownik współdzielony z wieloma językami, jak w mBERT) zmusza sieć do reprezentowania słów za pomocą bardzo drobnych cząstek (nawet pojedynczych liter), co utrudnia naukę zależności długodystansowych i zwiększa podatność na błędy ortograficzne czy szum.

### Zadanie 5: N-gramy i kolokacje

Analiza n-gramowa pozwala odkryć powtarzalne struktury składniowe występujące w badanym korpusie. Implementujemy analizę dla związków dwuwyrazowych (Bigramy) i trzywyrazowych (Trigramy).

In [ ]:
from stop_words import get_stop_words

def generate_ngrams(tokens_list, n):
    return zip(*[tokens_list[i:] for i in range(n)])

# Część A: Bigramy surowe
raw_bigrams = Counter(generate_ngrams(all_tokens, 2))
print("--- TOP 20 NAJCZĘSTSZYCH BIGRAMÓW (SUROWE) ---")
for bg, count in raw_bigrams.most_common(20):
    print(f"{' '.join(bg):<20} : {count}")

# Część B: Trigramy surowe
raw_trigrams = Counter(generate_ngrams(all_tokens, 3))
print("\n--- TOP 15 NAJCZĘSTSZYCH TRIGRAMÓW (SUROWE) ---")
for tg, count in raw_trigrams.most_common(15):
    print(f"{' '.join(tg):<25} : {count}")

# Część C: Porównanie po usunięciu stop-słów
pl_stopwords = get_stop_words('pl')
filtered_tokens = [tok for tok in all_tokens if tok not in pl_stopwords and len(tok) > 1]

filtered_bigrams = Counter(generate_ngrams(filtered_tokens, 2))
print("\n--- TOP 20 NAJCZĘSTSZYCH BIGRAMÓW (PO USUNIĘCIU STOP-WORDS) ---")
for bg, count in filtered_bigrams.most_common(20):
    print(f"{' '.join(bg):<25} : {count}")

#### Wnioski z analizy n-gramowej:
1. **Surowe n-gramy:** Wersje bez czyszczenia ze słów funkcyjnych składają się niemal wyłącznie ze spójników i przyimków (np. *"w roku"*, *"i w"*, *"na terenie"*). Nie niosą one istotnej wartości informacyjnej o unikalnej tematyce tekstu.
2. **N-gramy przefiltrowane:** Po eliminacji polskich stop-słów algorytm natychmiastowo ujawnia rzeczywiste kolokacje encji encyklopedycznych charakterystycznych dla Wikipedii (np. *"ii wojny"*, *"wojny światowej"*, *"wieku prawami"*, *"miejscowości położonej"*), co potwierdza poprawność zrealizowanego potoku przetwarzania wstępnego (Preprocessing Pipeline).